In [ ]:
import torch, os, math
import torchvision as tv
import torchvision.transforms.functional as tvf
from torchvision import io
import matplotlib.pyplot as plt
from torch.utils.cpp_extension import load_inline


In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING']='1'
os.environ['TORCH_CUDA_ARCH_LIST'] = "6.1"

In [ ]:
def load_cuda(cuda_src, cpp_src, funcs, opt=False, verbose=False):
    return load_inline(cuda_sources=[cuda_src], cpp_sources=[cpp_src], functions=funcs,
                       extra_cuda_cflags=["-O2"] if opt else [], verbose=verbose, name="inline_ext")

In [ ]:
cuda_begin = r'''
#include <torch/extension.h>
#include <stdio.h>
#include <c10/cuda/CUDAException.h>

#define CHECK_CUDA(x) TORCH_CHECK(x.device().is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

inline unsigned int cdiv(unsigned int a, unsigned int b) { return (a + b - 1) / b;}
'''

In [ ]:
cuda_src = cuda_begin + r'''
__global__ void helloworld_kernel() {
    printf("hello world!\n");
}

void helloworld() {
    helloworld_kernel<<<1, 1>>>();
    C10_CUDA_KERNEL_LAUNCH_CHECK();
}'''

In [ ]:
cpp_src = "void helloworld();"
print(torch.cuda.get_device_capability())

In [ ]:
#module = load_cuda(cuda_src, cpp_src, ['helloworld'], verbose=True)
print(torch.cuda.is_available())

In [ ]:
moudle = load_cuda(cuda_src, cpp_src, ['helloworld'], verbose=True)

In [ ]:
[o for o in dir(moudle) if o[0]!='_']

In [ ]:
%%time
moudle.helloworld()